In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import numpy as np
from typing import List, Dict, Any

# =========================================================================================
# 💡 데이터셋: camiller/final_korean_battles
# 📖 의미: 한국의 주요 전투(역사적 사건)에 대한 질의응답 및 지시문 데이터셋입니다.
# 🔎 목적: 사용자가 특정 지침(instruction)과 추가 정보(input)를 받으면,
#       그에 맞춰 어떤 형태의 답변(output)을 생성하는지 학습할 수 있는 프롬프트 데이터셋입니다.
# 🎯 초보자 실습 목표: 데이터셋의 구조를 이해하고, 마치 LLM처럼 세 가지 요소(지침, 입력, 출력)를 조합하여
#                     "완벽한 AI 질문지"를 스스로 만들어보는 과정을 실습합니다!
# =========================================================================================

# 상수 정의
DATASET_NAME = "camiller/final_korean_battles"
SAMPLE_COUNT = 10 # 실습을 위해 상위 10개만 사용합니다!

# 1. 사용 가능한 Config 목록 확인 (필수 과정)
print("======== 🚀 데이터셋 로딩 준비: Config 확인 ========")
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    # 기본 설정을 사용합니다.
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 목록 확인 중 오류 발생 또는 기본 설정만 사용합니다. ({e})")
    selected_config = None

# 2. 데이터 로딩 (스트리밍 최적화)
print("\n======== 💻 데이터셋 다운로드 및 준비 ========")

dataset = None
try:
    # 🚀 1차 시도: 무제한 스트리밍 모드로 로드 시도 (메모리 효율 최적화)
    print("✨ 1. 스트리밍 모드 (streaming=True)로 데이터셋 로드 시도 중...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("🎉 성공! 스트리밍 모드로 데이터셋을 메모리 효율적으로 로드했습니다.")

except Exception as e:
    # 😭 스트리밍 모드 실패 시: 소량의 데이터를 일반 모드로 다운로드하여 진행
    print(f"⚠️ 스트리밍 로드 실패 감지 ({e}). 소량의 데이터를 일반 모드로 대체 로드합니다.")
    try:
        # 주의: 실제 에러 처리에서는 'train' 스플릿이 아닌 소규모 스플릿을 지정할 수 있습니다.
        dataset = load_dataset(DATASET_NAME, split='train[:100]', streaming=False)
        print("✅ 성공! 소량의 데이터를 다운로드하여 안정적으로 진행할 수 있게 되었습니다.")
    except Exception as e_fallback:
        print(f"🚨 최종 로딩 실패. 데이터셋 로딩에 실패했습니다: {e_fallback}")
        exit()


# 3. 샘플 데이터 추출 및 준비 (가장 중요한 단계!)
print("\n======== 💾 데이터 추출 및 샘플링 ========")
sample_data_list: List[Dict[str, str]] = []

# ✨ 데이터셋이 take() 메소드를 지원하는지 확인합니다.
if hasattr(dataset, "take"):
    print(f"🔗 'take' 메소드 발견! 상위 {SAMPLE_COUNT}개의 샘플을 Iterator로 가져옵니다.")
    # 패턴 9번 준수: 스트리밍 또는 일반 데이터셋 모두 take()를 사용합니다.
    sample_data_iterator = dataset.take(SAMPLE_COUNT)
    # 패턴 16번 준수: 반복자(iterator)를 리스트로 변환합니다.
    sample_data_list = list(sample_data_iterator)
else:
    print("❌ 'take' 메소드를 찾을 수 없습니다. 데이터셋 전체를 리스트로 변환합니다. (권장되지 않음)")
    # fallback: 이 경우는 거의 발생하지 않아야 합니다.
    sample_data_list = list(dataset)


print(f"\n⭐ 총 {len(sample_data_list)}개의 샘플을 성공적으로 준비했습니다. 이제 실습을 시작합니다!")

# -------------------------------------------------------------------------------------
# 🌈 초보자를 위한 AI 실습: "만능 프롬프트 생성기"
# -------------------------------------------------------------------------------------

def generate_prompt_template(sample: Dict[str, str]) -> str:
    """
    하나의 데이터 샘플(Dict)을 받아서, LLM에 넣을 완벽한 프롬프트 템플릿을 만들어줍니다.
    (Instruction + Input)
    """
    instruction = sample.get("instruction", "")
    input_context = sample.get("input", "")
    
    # 주석을 통해 어떤 키를 사용했는지 명시합니다.
    # 패턴 1-1 준수: features 접근 대신 직접 키를 사용합니다.
    
    template = f"\n[💡 사용자 지침]: {instruction}\n"
    if input_context:
        template += f"[📚 추가 컨텍스트]: {input_context}\n"
    else:
        template += "[📚 추가 컨텍스트]: 없음\n"
    
    return template

def simulate_completion(sample: Dict[str, str]) -> str:
    """
    데이터셋의 output을 사용하여, AI가 어떻게 응답할지 시뮬레이션 합니다.
    """
    output = sample.get("output", "")
    
    if output.strip() == "":
        return "🤖 AI가 생성할 답변이 비어 있습니다. 🤔"
    
    return f"✅ AI 답변 시뮬레이션 결과:\n===\n{output}\n==="

print("\n\n=====================================================================")
print("✨ 실습 1: 🤖 만능 프롬프트 템플릿 조합 능력 테스트")
print("=====================================================================")
print("우리가 가진 데이터는 '지침', '입력', '출력' 3가지 조각입니다.")
print("이 3가지 조각을 잘 조합하여 최고의 질문을 만들 수 있는지 확인해봅시다!")

# 실습 로직 수행
for i, sample in enumerate(sample_data_list):
    print(f"\n--- [샘플 {i+1}/{SAMPLE_COUNT}] ---")
    
    # 1. 프롬프트 조합 (Combine)
    prompt = generate_prompt_template(sample)
    print("⭐ [생성된 완벽한 질문 프롬프트]:")
    print(prompt)
    
    # 2. 예상 답변 확인 (Predict)
    completion = simulate_completion(sample)
    print(completion)


# -------------------------------------------------------------------------------------
# 📚 데이터 분석 실습: '지침'의 특징 분석 (핵심 키워드 탐색)
# -------------------------------------------------------------------------------------

print("\n\n=====================================================================")
print("🔍 실습 2: 📖 '지침(Instruction)'의 특징 분석 및 키워드 탐색")
print("=====================================================================")

instruction_texts = [sample.get("instruction", "") for sample in sample_data_list]

# 간단한 키워드 카운터 구현 (Counter 역할을 수행)
keyword_counts: Dict[str, int] = {}

for instruction in instruction_texts:
    # 간단한 토큰화: 쉼표나 공백으로 분리
    tokens = instruction.lower().replace(",", " ").split()
    
    # 빈 문자열 및 너무 짧은 단어는 건너뜁니다.
    for token in tokens:
        if len(token) > 1 and token.strip() != "the":
            # 키워드가 이미 카운트 되어 있다면 1 증가, 아니라면 1로 초기화
            keyword_counts[token] = keyword_counts.get(token, 0) + 1

# 상위 5개 키워드 출력
sorted_keywords = sorted(keyword_counts.items(), key=lambda item: item[1], reverse=True)

print("\n📊 분석 결과: 가장 자주 등장한 상위 5개 키워드:")
print("===================================================")

for keyword, count in sorted_keywords[:5]:
    print(f"🔑 '{keyword}': {count}회 등장 (이 데이터셋의 주요 관심사입니다!)")

print("\n✨ 실습 완료! 축하드립니다!")
print("당신은 이제 데이터셋의 구조를 이해하고, LLM이 어떤 데이터를 원하며, 그 데이터를 어떻게 가공할지 파악하는 능력을 갖추었습니다.")